# 02 Interface Integration Test (synthetic only, no dataset)

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
from src.data_types import LiDARFrame
from src.feature_adapter import build_perception_from_labeled_points, adapt_perception_to_regions
from src.interface_validator import validate_lidar_frame, validate_perception_result, validate_region_features
from src.stage1_engines import ImportanceEngine, ResolutionEngine, AdaptiveMapper2_5D
print('imports OK')

imports OK


In [2]:
# Cell 2: synthetic LiDAR (controlled primitives, NOT real data)
rng = np.random.default_rng(42)
ped = np.column_stack([70 + rng.normal(0, 0.2, 200), 2 + rng.normal(0, 0.2, 200), rng.normal(0.9, 0.1, 200), rng.uniform(0.4, 0.7, 200)])
road = np.column_stack([70 + rng.uniform(-4, 4, 300), -6 + rng.uniform(-2, 2, 300), rng.normal(0, 0.02, 300), rng.uniform(0.2, 0.5, 300)])
pts = np.vstack([ped, road])
labels = np.array(['pedestrian']*len(ped) + ['road']*len(road))
print(pts.shape, 'synthetic Nx4 OK')

(500, 4) synthetic Nx4 OK


In [3]:
# Cell 3: LiDARFrame
frame = LiDARFrame(frame_id='synth-70m', timestamp=0.0, points=pts)
validate_lidar_frame(frame)
print('LiDARFrame valid:', frame.points.shape)

LiDARFrame valid: (500, 4)


In [4]:
# Cell 4: PerceptionResult (synthetic perception values)
conf = np.array([0.85]*len(ped) + [0.95]*len(road))
pr = build_perception_from_labeled_points('synth-70m', pts, labels, conf)
print('PerceptionResult:', pr.points.shape, len(pr.semantic_labels))

PerceptionResult: (500, 4) 500


In [5]:
# Cell 5: validate PerceptionResult
validate_perception_result(pr)
print('PerceptionResult valid')

PerceptionResult valid


In [6]:
# Cell 6: PerceptionResult -> RegionFeatures
regions = adapt_perception_to_regions(pr, bin_m=4.0, min_points=15)
print('regions:', len(regions))
for r in regions[:6]:
    print(r.region_id, r.semantic_label, round(r.distance, 1), r.point_count)

regions: 4
0 road 67.4 72
1 road 70.2 147
2 pedestrian 70.0 200
3 road 73.3 81


In [7]:
# Cell 7: validate RegionFeatures
for r in regions:
    validate_region_features(r)
print('all RegionFeatures valid')

all RegionFeatures valid


In [8]:
# Cell 8: existing Importance Engine (unchanged)
imp = ImportanceEngine()
ix = np.floor(pts[:, 0]/4.0).astype(int)
iy = np.floor(pts[:, 1]/4.0).astype(int)
_, inv = np.unique(np.column_stack([ix, iy]), axis=0, return_inverse=True)
full = [r.to_legacy_region(pts[inv == r.region_id]) for r in regions if (inv == r.region_id).sum() > 0]
scored = [imp.score_region(lr) for lr in full]
for s in scored[:6]:
    print(s.region_id, round(s.base_importance, 3), round(s.safe_importance, 3))

0 0.139 0.164
1 0.131 0.156
2 0.578 0.653
3 0.121 0.146


In [9]:
# Cell 9: existing Resolution Engine (unchanged)
res = ResolutionEngine()
for s in scored:
    m, lvl = res.select(s.safe_importance)
    s.selected_resolution_m, s.resolution_level = m, lvl
for s in scored[:6]:
    print(s.region_id, round(s.safe_importance, 3), s.selected_resolution_m, s.resolution_level)

0 0.164 0.5 coarse (50 cm)
1 0.156 0.5 coarse (50 cm)
2 0.653 0.1 medium_fine (10 cm)
3 0.146 0.5 coarse (50 cm)


In [10]:
# Cell 10: existing Adaptive 2.5D Mapper (unchanged)
mapper = AdaptiveMapper2_5D(imp, res)
mres = mapper.map_regions_legacy(full)
print('cells:', len(mres.cells), 'points:', mres.n_points, 'time_s:', round(mres.elapsed_s, 3))

cells: 179 points: 500 time_s: 0.029


In [11]:
# Cell 11: interface proof table
import pandas as pd
rows = [{'region_id': full[i].region_id, 'label': full[i].semantic_class, 'dist_m': round(full[i].distance_m, 1), 'base': round(s.base_importance, 3), 'safe': round(s.safe_importance, 3), 'res_m': s.selected_resolution_m} for i, s in enumerate(scored)]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

 region_id      label  dist_m  base  safe  res_m
         0       road    67.4 0.139 0.164    0.5
         1       road    70.2 0.131 0.156    0.5
         2 pedestrian    70.0 0.578 0.653    0.1
         3       road    73.3 0.121 0.146    0.5


In [12]:
# Cell 12: semantic-vs-distance USP (measured, both near 70 m)
print(df.sort_values('dist_m').to_string(index=False))
print('Result: pedestrian near 70 m gets finer resolution than road near 70 m.')
print('Importance-driven, computed above from synthetic data.')

 region_id      label  dist_m  base  safe  res_m
         0       road    67.4 0.139 0.164    0.5
         2 pedestrian    70.0 0.578 0.653    0.1
         1       road    70.2 0.131 0.156    0.5
         3       road    73.3 0.121 0.146    0.5
Result: pedestrian near 70 m gets finer resolution than road near 70 m.
Importance-driven, computed above from synthetic data.
